In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('2019-Oct_smartphone_preprocessed_final.csv')

df.head()

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session,is_price_error,hour,day_of_week,day_name,time_segment,is_outlier,is_purchase,is_cart
0,2019-10-01 09:00:04,view,1004237,-1769995873,electronics.smartphone,apple,1081.98,535871217,c6bd7419-2748-4c56-95b4-8cec9ff8b80d,False,9,1,Tuesday,오전 (6-12시),False,0,0
1,2019-10-01 09:00:11,view,1004545,-1769995873,electronics.smartphone,huawei,566.01,537918940,406c46ed-90a4-4787-a43b-59a410c1a5fb,False,9,1,Tuesday,오전 (6-12시),False,0,0
2,2019-10-01 09:00:11,view,1005011,-1769995873,electronics.smartphone,samsung,900.64,530282093,50a293fb-5940-41b2-baf3-17af0e812101,False,9,1,Tuesday,오전 (6-12시),False,0,0
3,2019-10-01 09:00:19,view,1005135,-1769995873,electronics.smartphone,apple,1747.79,535871217,c6bd7419-2748-4c56-95b4-8cec9ff8b80d,False,9,1,Tuesday,오전 (6-12시),False,0,0
4,2019-10-01 09:00:20,view,1003306,-1769995873,electronics.smartphone,apple,588.77,555446831,6ec635da-ea15-4a5d-96b4-c8ca9d38f89f,False,9,1,Tuesday,오전 (6-12시),False,0,0


In [2]:
df = df.sort_values(['user_session', 'event_time']).copy()

In [3]:
grouped = df.groupby(['user_session', 'product_id'])

In [4]:
event_first_time = (
    df
    .groupby(['user_session', 'product_id', 'event_type'])['event_time']
    .min()
    .unstack()
    .reset_index()
)

for col in ['view', 'cart', 'purchase']:
    if col not in event_first_time.columns:
        event_first_time[col] = pd.NaT

# 존재 여부 컬럼 따로 생성
event_first_time['has_view'] = event_first_time['view'].notna()
event_first_time['has_cart'] = event_first_time['cart'].notna()
event_first_time['has_purchase'] = event_first_time['purchase'].notna()

# 순서 검증
event_first_time['view_to_cart'] = (
    event_first_time['has_view'] &
    event_first_time['has_cart'] &
    (event_first_time['view'] < event_first_time['cart'])
)

event_first_time['view_to_cart_to_purchase'] = (
    event_first_time['has_view'] &
    event_first_time['has_cart'] &
    event_first_time['has_purchase'] &
    (event_first_time['view'] < event_first_time['cart']) &
    (event_first_time['cart'] < event_first_time['purchase'])
)

funnel_df = event_first_time[[
    'user_session',
    'product_id',
    'has_view',
    'view_to_cart',
    'view_to_cart_to_purchase'
]].rename(columns={
    'has_view': 'view'
})

funnel_df['segment'] = np.select(
    [
        funnel_df['view_to_cart_to_purchase'],  # 완전 전환
        
        funnel_df['view_to_cart'] & ~funnel_df['view_to_cart_to_purchase'],  # cart까지 갔다가 이탈
        
        funnel_df['view'] & ~funnel_df['view_to_cart']  # view만 하고 이탈
    ],
    [
        'conversion',
        'view_cart_drop',
        'view_only_drop'
    ],
    default='etc'
)

In [5]:
segment_summary = (
    funnel_df['segment']
    .value_counts()
    .reset_index()
)

segment_summary.columns = ['segment', 'count']

segment_summary['ratio'] = (
    segment_summary['count'] / segment_summary['count'].sum() * 100
)

segment_summary

,segment,count,ratio
0,view_only_drop,6695206,94.836909
1,conversion,190540,2.698980
2,view_cart_drop,172623,2.445187
3,etc,1336,0.018924


In [6]:
funnel_df['segment'].value_counts(normalize=True) * 100

segment
view_only_drop    94.836909
conversion         2.698980
view_cart_drop     2.445187
etc                0.018924
Name: proportion, dtype: float64

In [7]:
funnel_summary = pd.DataFrame({
    'step': ['view', 'view_to_cart', 'view_to_cart_to_purchase'],
    'count': [
        funnel_df['view'].sum(),
        funnel_df['view_to_cart'].sum(),
        funnel_df['view_to_cart_to_purchase'].sum()
    ]
})

funnel_summary

,step,count
0,view,7058369
1,view_to_cart,363163
2,view_to_cart_to_purchase,190540


In [8]:
# 1. 구매 전환 케이스
conversion_df = funnel_df[
    funnel_df['segment'] == 'conversion'
].copy()

# 2. view만 하고 이탈한 케이스
view_only_drop_df = funnel_df[
    funnel_df['segment'] == 'view_only_drop'
].copy()

# 3. view → cart 후 구매 없이 이탈한 케이스
view_cart_drop_df = funnel_df[
    funnel_df['segment'] == 'view_cart_drop'
].copy()

In [9]:
df_segment = df.merge(
    funnel_df[['user_session', 'product_id', 'segment']],
    on=['user_session', 'product_id'],
    how='left'
)

df_segment.head()

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session,is_price_error,hour,day_of_week,day_name,time_segment,is_outlier,is_purchase,is_cart,segment
0,2019-10-31 15:23:12,view,1005115,-1769995873,electronics.smartphone,apple,955.84,513782162,00000056-a206-40dd-b174-a072550fa38c,False,15,3,Thursday,오후 (12-18시),False,0,0,view_only_drop
1,2019-10-31 15:23:52,view,1005105,-1769995873,electronics.smartphone,apple,1349.46,513782162,00000056-a206-40dd-b174-a072550fa38c,False,15,3,Thursday,오후 (12-18시),False,0,0,view_only_drop
2,2019-10-31 15:25:30,view,1005105,-1769995873,electronics.smartphone,apple,1349.46,513782162,00000056-a206-40dd-b174-a072550fa38c,False,15,3,Thursday,오후 (12-18시),False,0,0,view_only_drop
3,2019-10-31 15:26:58,view,1004858,-1769995873,electronics.smartphone,samsung,131.53,513782162,00000056-a206-40dd-b174-a072550fa38c,False,15,3,Thursday,오후 (12-18시),False,0,0,view_only_drop
4,2019-10-31 15:28:21,view,1005104,-1769995873,electronics.smartphone,apple,993.27,513782162,00000056-a206-40dd-b174-a072550fa38c,False,15,3,Thursday,오후 (12-18시),False,0,0,view_only_drop


이탈 집단은 view 시점의 price / price 이상치 처리..

In [10]:
segment_price_detail = (
    df_segment[df_segment['event_type'] == 'view']
    .groupby('segment')
    .agg(
        view_count=('price', 'count'),
        avg_price=('price', 'mean'),
        median_price=('price', 'median'),
        min_price=('price', 'min'),
        max_price=('price', 'max')
    )
    .reset_index()
)

segment_price_detail

,segment,view_count,avg_price,median_price,min_price,max_price
0,conversion,468196,429.158637,250.78,0.0,2110.45
1,view_cart_drop,452229,425.895104,250.69,0.0,2110.45
2,view_only_drop,9690795,478.860301,287.97,0.0,2110.45


In [11]:
conversion_purchase_price = (
    df_segment[
        (df_segment['segment'] == 'conversion') &
        (df_segment['event_type'] == 'purchase')
    ]
    .groupby('segment')
    .agg(
        purchase_count=('price', 'count'),
        avg_purchase_price=('price', 'mean'),
        median_purchase_price=('price', 'median'),
        min_purchase_price=('price', 'min'),
        max_purchase_price=('price', 'max')
    )
    .reset_index()
)

conversion_purchase_price

,segment,purchase_count,avg_purchase_price,median_purchase_price,min_purchase_price,max_purchase_price
0,conversion,205941,432.636572,250.82,38.3,2110.45


세그먼트 별 브랜드 분포

In [12]:
segment_brand = (
    df_segment[df_segment['event_type'] == 'view']
    .groupby(['segment', 'brand'])
    .size()
    .reset_index(name='count')
    .sort_values(['segment', 'count'], ascending=[True, False])
)

segment_brand.head(20)

,segment,brand,count
19,conversion,samsung,213862
0,conversion,apple,143832
25,conversion,xiaomi,49334
9,conversion,huawei,35436
17,conversion,oppo,18812
24,conversion,vivo,2829
13,conversion,meizu,755
7,conversion,honor,631
14,conversion,nokia,611
23,conversion,unknown,402


In [13]:
segment_brand['segment_total'] = (
    segment_brand.groupby('segment')['count'].transform('sum')
)

segment_brand['ratio'] = (
    segment_brand['count'] / segment_brand['segment_total'] * 100
)

segment_brand.sort_values(['segment', 'ratio'], ascending=[True, False]).head(30)

,segment,brand,count,segment_total,ratio
19,conversion,samsung,213862,468196,45.677878
0,conversion,apple,143832,468196,30.720467
25,conversion,xiaomi,49334,468196,10.537040
9,conversion,huawei,35436,468196,7.568625
17,conversion,oppo,18812,468196,4.017975
24,conversion,vivo,2829,468196,0.604234
13,conversion,meizu,755,468196,0.161257
7,conversion,honor,631,468196,0.134773
14,conversion,nokia,611,468196,0.130501
23,conversion,unknown,402,468196,0.085861


상위 구매 / 이탈집단 상위 조회

In [14]:
segment_product_purchase = (
    df_segment[
        (df_segment['segment'] == 'conversion') &
        (df_segment['event_type'] == 'purchase')
    ]
    .groupby(['segment', 'product_id'])
    .agg(
        revenue=('price', 'sum'),
        purchase_count=('price', 'count')
    )
    .reset_index()
    .sort_values('revenue', ascending=False)
)

segment_product_purchase.head(10)

,segment,product_id,revenue,purchase_count
412,conversion,1005115,6039401.59,6118
402,conversion,1005105,5911730.57,4209
129,conversion,1004249,4082348.66,5511
246,conversion,1004767,3661407.61,14699
432,conversion,1005135,3258793.72,1883
8,conversion,1002544,3128714.21,6797
289,conversion,1004856,2850227.54,21711
1,conversion,1002524,2180312.37,4080
298,conversion,1004870,2074080.18,7271
22,conversion,1003306,1662802.74,2850


In [15]:
segment_product_purchase_count = (
    df_segment[
        (df_segment['segment'] == 'conversion') &
        (df_segment['event_type'] == 'purchase')
    ]
    .groupby(['segment', 'product_id'])
    .agg(
        revenue=('price', 'sum'),
        purchase_count=('price', 'count')
    )
    .reset_index()
    .sort_values('purchase_count', ascending=False)
)

segment_product_purchase_count.head(10)

,segment,product_id,revenue,purchase_count
289,conversion,1004856,2850227.54,21711
246,conversion,1004767,3661407.61,14699
274,conversion,1004833,1527182.10,8877
298,conversion,1004870,2074080.18,7271
8,conversion,1002544,3128714.21,6797
412,conversion,1005115,6039401.59,6118
277,conversion,1004836,1270714.20,5553
129,conversion,1004249,4082348.66,5511
301,conversion,1004873,1654218.86,4414
402,conversion,1005105,5911730.57,4209


In [16]:
segment_product_view = (
    df_segment[
        (df_segment['segment'].isin(['view_only_drop', 'view_cart_drop'])) &
        (df_segment['event_type'] == 'view')
    ]
    .groupby(['segment', 'product_id'])
    .agg(
        view_count=('product_id', 'size'),
        avg_price=('price', 'mean'),
        median_price=('price', 'median')
    )
    .reset_index()
    .sort_values(['segment', 'view_count'], ascending=[True, False])
)

segment_product_view.head(20)

,segment,product_id,view_count,avg_price,median_price
305,view_cart_drop,1004856,33186,131.296450,131.530
262,view_cart_drop,1004767,28119,248.995627,250.130
314,view_cart_drop,1004870,15148,285.296533,285.400
290,view_cart_drop,1004833,14329,171.994464,172.150
134,view_cart_drop,1004249,12421,740.788001,741.060
429,view_cart_drop,1005115,11893,987.867567,992.050
249,view_cart_drop,1004741,11843,190.084551,190.220
7,view_cart_drop,1002544,11104,460.302121,460.110
293,view_cart_drop,1004836,10757,228.601138,229.420
247,view_cart_drop,1004739,9853,191.843673,190.210


세그먼트 별 조회 상품수

In [17]:
views_segment = df_segment[df_segment['event_type'] == 'view'].copy()

session_view_depth_segment = (
    views_segment.groupby(['segment', 'user_session'])
    .agg(
        view_event_count=('product_id', 'size'),
        unique_view_products=('product_id', 'nunique')
    )
    .reset_index()
)

segment_view_depth = (
    session_view_depth_segment.groupby('segment')
    .agg(
        avg_view_events_per_session=('view_event_count', 'mean'),
        median_view_events_per_session=('view_event_count', 'median'),
        avg_unique_products_per_session=('unique_view_products', 'mean'),
        median_unique_products_per_session=('unique_view_products', 'median')
    )
    .reset_index()
)

segment_view_depth

,segment,avg_view_events_per_session,median_view_events_per_session,avg_unique_products_per_session,median_unique_products_per_session
0,conversion,2.608407,2.0,1.061534,1.0
1,view_cart_drop,2.792675,2.0,1.066008,1.0
2,view_only_drop,3.333063,2.0,2.302757,1.0


세그먼트별 동일 세션 내 모델 비교 깊이

In [18]:
segment_compare_depth = (
    views_segment.groupby(['segment', 'user_session'])
    .agg(
        unique_models_compared=('product_id', 'nunique')
    )
    .reset_index()
)

segment_compare_summary = (
    segment_compare_depth.groupby('segment')
    .agg(
        avg_model_compare_depth=('unique_models_compared', 'mean'),
        median_model_compare_depth=('unique_models_compared', 'median'),
        max_model_compare_depth=('unique_models_compared', 'max')
    )
    .reset_index()
)

segment_compare_ratio = (
    segment_compare_depth.groupby('segment')
    .apply(lambda g: pd.Series({
        'compare_2plus_session_ratio': (g['unique_models_compared'] >= 2).mean() * 100,
        'compare_3plus_session_ratio': (g['unique_models_compared'] >= 3).mean() * 100
    }))
    .reset_index()
)

segment_compare_summary = segment_compare_summary.merge(
    segment_compare_ratio,
    on='segment',
    how='left'
)

segment_compare_summary

,segment,avg_model_compare_depth,median_model_compare_depth,max_model_compare_depth,compare_2plus_session_ratio,compare_3plus_session_ratio
0,conversion,1.061534,1.0,10,5.232458,0.706983
1,view_cart_drop,1.066008,1.0,12,5.565230,0.765744
2,view_only_drop,2.302757,1.0,120,41.787923,24.831761


In [19]:
final_segment_compare = (
    segment_price_detail
    .merge(segment_view_depth, on='segment', how='left')
    .merge(segment_compare_summary, on='segment', how='left')
)

final_segment_compare

,segment,view_count,avg_price,median_price,min_price,max_price,avg_view_events_per_session,median_view_events_per_session,avg_unique_products_per_session,median_unique_products_per_session,avg_model_compare_depth,median_model_compare_depth,max_model_compare_depth,compare_2plus_session_ratio,compare_3plus_session_ratio
0,conversion,468196,429.158637,250.78,0.0,2110.45,2.608407,2.0,1.061534,1.0,1.061534,1.0,10,5.232458,0.706983
1,view_cart_drop,452229,425.895104,250.69,0.0,2110.45,2.792675,2.0,1.066008,1.0,1.066008,1.0,12,5.565230,0.765744
2,view_only_drop,9690795,478.860301,287.97,0.0,2110.45,3.333063,2.0,2.302757,1.0,2.302757,1.0,120,41.787923,24.831761
